# Olist Late Delivery Prediction — Improved Notebook

This notebook is an updated copy of `olist_late_delivery_analysis.ipynb`.

It uses the improved project scripts:

- `make_dataset_v2.py` for dataset creation and feature engineering
- `train_model.py` for feature selection, pipeline creation, threshold selection, and evaluation
- `tune_threshold.py` for threshold trade-off analysis from saved predictions

Main goal: predict `late_probability`, the probability that an order will be delivered after the estimated delivery date.

## 1. Setup

The notebook imports reusable functions from the Python scripts so that notebook logic stays aligned with the production-style pipeline.

In [1]:
from datetime import datetime, timezone
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split

from make_dataset_v2 import build_dataset, OUTPUT_PATH
from train_model import (
    RANDOM_STATE,
    MODEL_PATH,
    PREDICTIONS_PATH,
    METADATA_PATH,
    THRESHOLD_REPORT_PATH,
    get_feature_columns,
    make_pipeline,
    find_best_threshold,
    evaluate_split,
)

pd.set_option("display.max_columns", 100)
print("scikit-learn version:", sklearn.__version__)

scikit-learn version: 1.9.0


## 2. Build or load the modelling dataset

The improved dataset builder adds safer script-relative paths and extra features such as:

- `is_weekend_purchase`
- `has_approval_timestamp`
- `main_seller_city`
- `same_city_customer_seller`

In [2]:
if OUTPUT_PATH.exists():
    df = pd.read_csv(OUTPUT_PATH)
    print(f"Loaded existing dataset: {OUTPUT_PATH}")
else:
    df = build_dataset()
    df.to_csv(OUTPUT_PATH, index=False)
    print(f"Built and saved dataset: {OUTPUT_PATH}")

print("Dataset shape:", df.shape)
display(df.head())

Loaded existing dataset: C:\Users\sobhan\Downloads\project1-mlops\model_dataset_v2.csv
Dataset shape: (96476, 39)


,is_late,purchase_hour,purchase_dayofweek,purchase_month,is_weekend_purchase,estimated_delivery_days,has_approval_timestamp,approval_delay_hours,num_items,total_price,avg_price,max_price,total_freight,avg_freight,max_freight,num_sellers,num_products,num_product_categories,avg_product_weight_g,max_product_weight_g,avg_product_volume_cm3,max_product_volume_cm3,seller_zip_code_prefix,num_seller_states,main_seller_city,main_seller_state,main_product_category,freight_price_ratio,payment_value,payment_installments,num_payment_types,main_payment_type,customer_zip_code_prefix,customer_city,customer_state,same_state_customer_seller,same_city_customer_seller,zip_prefix_diff,customer_seller_distance_km
0,0,10,0,10,0,15.544063,1,0.178333,1,29.99,29.99,29.99,8.72,8.72,8.72,1,1,1,500.0,500.0,1976.0,1976.0,9350.0,1,maua,SP,housewares,0.290764,38.71,1.0,2.0,voucher,3149,sao paulo,SP,1,0,6201.0,NaN
1,0,20,1,7,0,19.137766,1,30.713889,1,118.70,118.70,118.70,22.76,22.76,22.76,1,1,1,400.0,400.0,4693.0,4693.0,31570.0,1,belo horizonte,SP,perfumery,0.191744,141.46,1.0,1.0,boleto,47813,barreiras,BA,0,0,16243.0,NaN
2,0,8,2,8,0,26.639711,1,0.276111,1,159.90,159.90,159.90,19.22,19.22,19.22,1,1,1,420.0,420.0,9576.0,9576.0,14840.0,1,guariba,SP,auto,0.120200,179.12,3.0,1.0,credit_card,75265,vianopolis,GO,0,0,60425.0,NaN
3,0,19,5,11,1,26.188819,1,0.298056,1,45.00,45.00,45.00,27.20,27.20,27.20,1,1,1,450.0,450.0,6000.0,6000.0,31842.0,1,belo horizonte,MG,pet_shop,0.604444,72.20,1.0,1.0,credit_card,59296,sao goncalo do amarante,RN,0,0,27454.0,NaN
4,0,21,1,2,0,12.112049,1,1.030556,1,19.90,19.90,19.90,8.72,8.72,8.72,1,1,1,250.0,250.0,11475.0,11475.0,8752.0,1,mogi das cruzes,SP,stationery,0.438191,28.62,1.0,1.0,credit_card,9195,santo andre,SP,1,0,443.0,NaN


## 3. Target distribution and feature selection

The target is `is_late`. Leakage and raw ID columns are excluded by `get_feature_columns()` from `train_model.py`.

In [3]:
target_col = "is_late"
if target_col not in df.columns:
    raise ValueError(f"Target column {target_col!r} was not found.")

numeric_features, categorical_features = get_feature_columns(df, target_col)
feature_cols = numeric_features + categorical_features

X = df[feature_cols]
y = df[target_col].astype(int)

print("Target counts:")
print(y.value_counts())
print("\nTarget ratio:")
print(y.value_counts(normalize=True))
print("\nNumber of features:", len(feature_cols))
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

Target counts:
is_late
0    88649
1     7827
Name: count, dtype: int64

Target ratio:
is_late
0    0.918871
1    0.081129
Name: proportion, dtype: float64

Number of features: 38
Numeric features: ['purchase_hour', 'purchase_dayofweek', 'purchase_month', 'is_weekend_purchase', 'estimated_delivery_days', 'has_approval_timestamp', 'approval_delay_hours', 'num_items', 'total_price', 'avg_price', 'max_price', 'total_freight', 'avg_freight', 'max_freight', 'num_sellers', 'num_products', 'num_product_categories', 'avg_product_weight_g', 'max_product_weight_g', 'avg_product_volume_cm3', 'max_product_volume_cm3', 'seller_zip_code_prefix', 'num_seller_states', 'freight_price_ratio', 'payment_value', 'payment_installments', 'num_payment_types', 'customer_zip_code_prefix', 'same_state_customer_seller', 'same_city_customer_seller', 'zip_prefix_diff', 'customer_seller_distance_km']
Categorical features: ['main_seller_city', 'main_seller_state', 'main_product_category', 'main_payment_type', 'custome

## 4. Train / Validation / Test split

The validation set is used only for threshold selection. The test set remains untouched until final evaluation.

In [4]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_train_full,
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (57885, 38)
Validation: (19295, 38)
Test: (19296, 38)


## 5. Train the model

The model pipeline uses:

- median imputation for numeric columns
- most-frequent imputation + ordinal encoding for categorical columns
- `HistGradientBoostingClassifier` for non-linear tabular modelling

In [5]:
clf = make_pipeline(numeric_features, categorical_features)

# Train on X_train and calibrate probabilities on X_val.
# New scikit-learn versions require an explicit train/calibration split
# train/calibration split to CalibratedClassifierCV instead.
X_calibration_fit = pd.concat([X_train, X_val], axis=0)
y_calibration_fit = pd.concat([y_train, y_val], axis=0)
train_idx = np.arange(len(X_train))
calibration_idx = np.arange(len(X_train), len(X_calibration_fit))

calibrator = CalibratedClassifierCV(
    clf,
    method="isotonic",
    cv=[(train_idx, calibration_idx)],
)

calibrator.fit(X_calibration_fit, y_calibration_fit)
val_proba = calibrator.predict_proba(X_val)[:, 1]
test_proba = calibrator.predict_proba(X_test)[:, 1]

calibration_summary = pd.DataFrame([
    {
        "split": "Validation",
        "rows": len(y_val),
        "actual_late_rate": float(y_val.mean()),
        "mean_predicted_late_probability": float(np.mean(val_proba)),
    },
    {
        "split": "Test",
        "rows": len(y_test),
        "actual_late_rate": float(y_test.mean()),
        "mean_predicted_late_probability": float(np.mean(test_proba)),
    },
])

print("Example calibrated late probabilities:")
print(test_proba[:10])
display(calibration_summary)

C:\Users\sobhan\AppData\Roaming\Python\Python312\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['customer_seller_distance_km']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\sobhan\AppData\Roaming\Python\Python312\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['customer_seller_distance_km']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\sobhan\AppData\Roaming\Python\Python312\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['customer_seller_distance_km']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Example calibrated late probabilities:
[0.01619778 0.03543307 0.08292282 0.02939255 0.02939255 0.03543307
 0.07350689 0.0166998  0.09375    0.04489623]


C:\Users\sobhan\AppData\Roaming\Python\Python312\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['customer_seller_distance_km']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


,split,rows,actual_late_rate,mean_predicted_late_probability
0,Validation,19295,0.081109,0.081109
1,Test,19296,0.081105,0.081189


## 6. Select a business-aware threshold

By default, the threshold maximizes validation F1. You can optionally enforce minimum precision or recall.

Examples:

- `MIN_RECALL = 0.50` if catching late orders is more important
- `MIN_PRECISION = 0.35` if reducing false alerts is more important

In [6]:
MIN_PRECISION = 0.0
MIN_RECALL = 0.0

best_threshold, threshold_metrics, threshold_report = find_best_threshold(
    y_val,
    val_proba,
    min_precision=MIN_PRECISION,
    min_recall=MIN_RECALL,
)

threshold_report.to_csv(THRESHOLD_REPORT_PATH, index=False)

print("Best threshold:", round(best_threshold, 4))
print("Threshold metrics:", threshold_metrics)
print("Saved threshold report:", THRESHOLD_REPORT_PATH)
display(threshold_report.sort_values("f1", ascending=False).head(10))

Best threshold: 0.2275
Threshold metrics: {'precision': 0.3473118279569892, 'recall': 0.412779552715655, 'f1': 0.3772262773717665, 'min_precision': 0.0, 'min_recall': 0.0, 'constraints_satisfied': True}
Saved threshold report: C:\Users\sobhan\Downloads\project1-mlops\threshold_report.csv


,threshold,precision,recall,f1,meets_constraints
24,0.227513,0.347312,0.412780,0.377226,True
23,0.180000,0.342932,0.418530,0.376978,True
22,0.179878,0.301247,0.493930,0.374244,True
21,0.177914,0.293881,0.512460,0.373544,True
20,0.166667,0.293601,0.513099,0.373488,True
25,0.257143,0.360862,0.385304,0.372682,True
19,0.166205,0.278747,0.551438,0.370307,True
26,0.268293,0.370346,0.362300,0.366279,True
27,0.285036,0.376121,0.348243,0.361646,True
18,0.127980,0.247881,0.616613,0.353609,True


## 7. Evaluate validation performance

The selected threshold is checked on validation first. The test set is not used for threshold selection and is evaluated only after fitting the final model. Because late delivery is imbalanced, the main metrics are PR-AUC, F1, precision, and recall rather than accuracy.

In [7]:
val_metrics = evaluate_split("Validation", y_val, val_proba, best_threshold)


Validation metrics
------------------
ROC-AUC: 0.8073
PR-AUC:  0.3163
Precision: 0.3473
Recall:    0.4128
F1:        0.3772

Confusion Matrix:
[[16516  1214]
 [  919   646]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9473    0.9315    0.9393     17730
           1     0.3473    0.4128    0.3772      1565

    accuracy                         0.8895     19295
   macro avg     0.6473    0.6722    0.6583     19295
weighted avg     0.8986    0.8895    0.8938     19295



## 8. Save model, predictions, and metadata

After threshold selection, the final model is refit on train + validation to use more data. The test `late_probability` values are then recomputed with this same `final_clf`, using the already-fixed validation threshold.

In [8]:
final_clf = make_pipeline(numeric_features, categorical_features)
final_clf.fit(X_train_full, y_train_full)

final_test_proba = final_clf.predict_proba(X_test)[:, 1]
test_metrics = evaluate_split("Test", y_test, final_test_proba, best_threshold)

joblib.dump(final_clf, MODEL_PATH)

predictions = X_test.copy()
predictions["actual_is_late"] = y_test.values
predictions["late_probability"] = final_test_proba
predictions["predicted_is_late"] = (final_test_proba >= best_threshold).astype(int)
predictions.to_csv(PREDICTIONS_PATH, index=False)

metadata = {
    "model_file": str(MODEL_PATH),
    "dataset": str(OUTPUT_PATH),
    "model_type": "HistGradientBoostingClassifier",
    "threshold_selection": "max_f1_on_validation_set",
    "threshold": best_threshold,
    "threshold_report": str(THRESHOLD_REPORT_PATH),
    "primary_metrics": ["pr_auc", "f1", "precision", "recall", "roc_auc"],
    "validation_threshold_metrics": threshold_metrics,
    "validation_metrics": val_metrics,
    "test_metrics": test_metrics,
    "target_distribution": y.value_counts().to_dict(),
    "feature_columns": feature_cols,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "random_state": RANDOM_STATE,
    "sklearn_version": sklearn.__version__,
    "trained_at_utc": datetime.now(timezone.utc).isoformat(),
}

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)

print("Saved model:", MODEL_PATH)
print("Saved predictions:", PREDICTIONS_PATH)
print("Saved metadata:", METADATA_PATH)

C:\Users\sobhan\AppData\Roaming\Python\Python312\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['customer_seller_distance_km']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\sobhan\AppData\Roaming\Python\Python312\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['customer_seller_distance_km']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(



Test metrics
------------
ROC-AUC: 0.7974
PR-AUC:  0.3117
Precision: 0.3569
Recall:    0.3291
F1:        0.3424

Confusion Matrix:
[[16803   928]
 [ 1050   515]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9412    0.9477    0.9444     17731
           1     0.3569    0.3291    0.3424      1565

    accuracy                         0.8975     19296
   macro avg     0.6490    0.6384    0.6434     19296
weighted avg     0.8938    0.8975    0.8956     19296

Saved model: C:\Users\sobhan\Downloads\project1-mlops\late_delivery_model.joblib
Saved predictions: C:\Users\sobhan\Downloads\project1-mlops\predictions.csv
Saved metadata: C:\Users\sobhan\Downloads\project1-mlops\model_metadata.json


## 9. Summary

This improved notebook is connected to the updated scripts and avoids duplicating too much modelling logic.

Recommended next improvements:

1. Run `python make_dataset_v2.py` and `python train_model.py` from terminal to reproduce outputs.
2. Try threshold constraints such as `--min-recall 0.50`.
3. Compare additional models such as LightGBM, XGBoost, or CatBoost.
4. Add better distance/geography features between customer and seller.
5. Calibrate probabilities if `late_probability` will be used directly for business decisions.